In [156]:
import cv2
import numpy as np
import math
from PIL import Image, ImageDraw


In [157]:
p1 = (-1, -1); p2 = (-1, -1)

In [158]:
def translate_rectangle(p1, p2, translation):
    new_p1 = (p1[0] + translation[0], p1[1] + translation[1])
    new_p2 = (p2[0] + translation[0], p2[1] + translation[1])
    return new_p1, new_p2

In [159]:
def create_white_background(width, height):
    return np.ones((height, width, 3), dtype=np.uint8) * 255

In [160]:
def rotate_rectangle(points, rotate_point, angle):
    angle_rad = math.radians(angle)
    translated_points = [(pt[0] - rotate_point[0], pt[1] - rotate_point[1]) for pt in points]
    rotated_points = []
    for pt in translated_points:
        x = pt[0] * math.cos(angle_rad) - pt[1] * math.sin(angle_rad)
        y = pt[0] * math.sin(angle_rad) + pt[1] * math.cos(angle_rad)
        rotated_points.append((x, y))
    final_points = [(pt[0] + rotate_point[0], pt[1] + rotate_point[1]) for pt in rotated_points]
    return final_points

In [161]:
def scale_rectangle(p1, p2, scale_factors):
    new_p2_x = int(p2[0] * scale_factors[0])
    new_p2_y = int(p2[1] * scale_factors[1])
    return (p1), (new_p2_x, new_p2_y)

In [162]:
def draw_rectangle(image, p1, p2, color):
    cv2.rectangle(image, p1, p2, color, 2)

In [163]:
def mouse_callback(event, x, y, flags, param):
    global p1, p2
    if event == cv2.EVENT_LBUTTONDOWN:
        p1 = (x, y)
    elif event == cv2.EVENT_LBUTTONUP:
        p2 = (x, y)
        cv2.rectangle(image, p1, p2, (0, 0, 255), 2)

In [164]:
image_width = 800
image_height = 600
image = create_white_background(image_width, image_height)
cv2.namedWindow("Rectangle_window")
cv2.setMouseCallback("Rectangle_window", mouse_callback)

In [165]:
def draw_rectangle1(image, points):
    cv2.polylines(image, np.int32([points]), True, (0, 0, 255), 2)

In [166]:
while True:
    cv2.imshow("Rectangle_window", image)
    key = cv2.waitKey(1) & 0xFF

    if key == ord("t"):
        x = int(input("Enter x trans: "))
        y = int(input("Enter y trans: "))
        p1, p2 = translate_rectangle(p1, p2, (x, y))
        image = create_white_background(image_width, image_height)
        draw_rectangle(image, p1, p2, (0, 0, 255))

    elif key == ord("r"):
        angle = float(input("Enter rotation angle in degrees: "))
        center_x = (p1[0] + p2[0]) // 2
        center_y = (p1[1] + p2[1]) // 2
        rectangle_points = [p1,(p1[0], p2[1]), p2,(p2[0], p1[1])]
        image = create_white_background(image_width, image_height)
        rotated_points  = rotate_rectangle(rectangle_points,(center_x,center_y),angle)
        draw_rectangle1(image, rotated_points)
        cv2.destroyAllWindows()
        image_cv = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
        cv2.imshow('Rrotated_rec', image_cv)
        cv2.destroyAllWindows()

    elif key == ord("s"):
        scale_x = float(input("Enter x scale factor: "))
        scale_y = float(input("Enter y scale factor: "))
        p1, p2 = scale_rectangle(p1, p2, (scale_x, scale_y))
        image = create_white_background(image_width, image_height)
        draw_rectangle(image, p1, p2, (0, 0, 255))

    elif key == ord("q"):
        break
cv2.destroyAllWindows()

In [167]:
import numpy as np
from scipy.optimize import minimize
# Given point pairs
points = {'A': ([1, 2], [6, 8]),
          'B': ([11, 13], [18, 13]),
          'C': ([21, 23], [28, 13]),
          'D': ([31, 37], [28, 33]),
          'E': ([51, 57], [68, 43])}
def geometric_transform(params, point_pair):
    # Extract parameters
    s, theta, tx, ty = params
    # Transformation matrix
    transform_matrix = np.array([[s * np.cos(theta), -s * np.sin(theta), tx],
                                [s * np.sin(theta), s * np.cos(theta), ty]])
    # Apply transformation to the original point
    transformed_point = np.dot(transform_matrix, np.array([point_pair[0][0], point_pair[0][1], 1]))
    # Calculate squared error
    error = np.sum((transformed_point[:2] - point_pair[1]) ** 2)
    return error
def total_mse(params):
    # Calculate total mean squared error for all point pairs
    total_error = 0
    for key, value in points.items():
        total_error += geometric_transform(params, value)
    return total_error / len(points)
# Initial guess for parameters [s, theta, tx, ty]
initial_guess = [1, 0, 0, 0]
# Minimize total mean squared error using optimization algorithm
result = minimize(total_mse, initial_guess, method='L-BFGS-B')
# Extract optimal parameters
optimal_params = result.x
print("Optimal Parameters:")
print("Scaling Factor (s):", optimal_params[0])
print("Rotation Angle (theta):", optimal_params[1])
print("Translation (tx, ty):", (optimal_params[2], optimal_params[3]))

Optimal Parameters:
Scaling Factor (s): 0.9283143488657434
Rotation Angle (theta): -0.25765666366566786
Translation (tx, ty): (2.7113899306319977, 3.7401591081978744)
